# ECG Signal Analysis Tutorial
## Digital Signal Processing Mini-Project

This notebook provides a comprehensive walkthrough of ECG signal analysis using DSP techniques.

**Course**: DJS22CEC802 (Theory) / DJS22CEL802 (Lab)  
**Author**: Nicky John  
**Date**: February 2026

---

## Table of Contents

1. [Introduction](#introduction)
2. [Setup](#setup)
3. [Unit 1: Signal Preprocessing](#unit1)
4. [Unit 2: Frequency Domain Analysis](#unit2)
5. [Units 5-6: Image Processing](#units5-6)
6. [Feature Extraction & Classification](#features)
7. [Complete Pipeline](#pipeline)
8. [Conclusion](#conclusion)

## 1. Introduction <a name="introduction"></a>

Electrocardiogram (ECG) signals record the electrical activity of the heart. Analyzing these signals helps detect:
- Arrhythmias (irregular heartbeats)
- Ischemia (reduced blood flow)
- Myocardial infarction (heart attack)

This project applies DSP concepts to process and analyze ECG signals:

### Theory Concepts (DJS22CEC802)
- **Unit 1**: Sampling, filtering, convolution
- **Unit 2**: DFT/FFT, wavelet transforms
- **Unit 3**: Overlap-Add method
- **Unit 5**: Histogram equalization, edge enhancement
- **Unit 6**: Edge detection, segmentation

### Lab Experiments (DJS22CEL802)
- **Exp 1-2**: Sampling and correlation
- **Exp 3-4**: DFT and FFT
- **Exp 7-10**: Image processing techniques

## 2. Setup <a name="setup"></a>

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
from ecg_preprocessing import ECGPreprocessor
from ecg_fft_dwt import FrequencyAnalyzer
from image_processing_ecg import ScalogramImageProcessor
from feature_extraction import ECGFeatureExtractor, ECGClassifier

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ All modules imported successfully!")

## 3. Unit 1: Signal Preprocessing <a name="unit1"></a>

### 3.1 Sampling and Reconstruction (Lab Exp 1)

**Theory**: According to the Nyquist-Shannon sampling theorem, a continuous signal can be perfectly reconstructed from its samples if:
$$f_s > 2 \cdot f_{max}$$

where $f_s$ is the sampling rate and $f_{max}$ is the highest frequency in the signal.

In [ ]:
# Initialize preprocessor
fs = 360  # Hz (MIT-BIH standard)
preprocessor = ECGPreprocessor(sampling_rate=fs)

# Generate synthetic ECG for demonstration
from scipy.signal.windows import gaussian

duration = 5  # seconds
t = np.arange(0, duration, 1/fs)
ecg = np.zeros_like(t)

# Create heartbeats
for beat_time in np.arange(0.5, duration, 0.8):
    beat_idx = int(beat_time * fs)
    if beat_idx < len(ecg) - 100:
        beat = gaussian(100, 10)
        ecg[beat_idx:beat_idx+100] += beat

# Add noise
noise = 0.1 * np.random.randn(len(ecg))
noisy_ecg = ecg + noise

print(f"Generated ECG: {duration} seconds @ {fs} Hz")
print(f"Total samples: {len(noisy_ecg)}")

In [ ]:
# Demonstrate sampling and reconstruction
downsampled, reconstructed, t_axis = preprocessor.downsample_and_reconstruct(
    noisy_ecg, downsample_factor=4
)

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

axes[0].plot(t_axis[:1000], noisy_ecg[:1000], 'b-', linewidth=0.8)
axes[0].set_title('Original Signal (360 Hz)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

t_down = t_axis[::4]
axes[1].stem(t_down[:250], downsampled[:250], linefmt='r-', markerfmt='ro', basefmt=' ')
axes[1].set_title('Downsampled Signal (90 Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t_axis[:1000], reconstructed[:1000], 'g-', linewidth=0.8)
axes[2].plot(t_axis[:1000], noisy_ecg[:1000], 'b--', linewidth=0.5, alpha=0.5, label='Original')
axes[2].set_title('Reconstructed Signal (interpolated back to 360 Hz)')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Amplitude')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Lab Exp 1: Sampling & Reconstruction demonstrated")

### 3.2 Linear Convolution for Filtering (Unit 1.2)

**Theory**: Linear convolution of signal $x[n]$ with impulse response $h[n]$:
$$y[n] = x[n] * h[n] = \sum_{k=-\infty}^{\infty} x[k] \cdot h[n-k]$$

We use this for FIR filtering to remove noise.

In [ ]:
# Apply bandpass filter (0.5-40 Hz)
filtered_ecg = preprocessor.filter_ecg(noisy_ecg, method='filtfilt')

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(t[:1000], noisy_ecg[:1000], 'b-', linewidth=0.8)
axes[0].set_title('Noisy ECG Signal')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t[:1000], filtered_ecg[:1000], 'g-', linewidth=0.8)
axes[1].set_title('Filtered ECG (0.5-40 Hz Bandpass)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Unit 1.2: Linear Convolution filtering applied")

### 3.3 R-Peak Detection via Cross-Correlation (Lab Exp 2)

**Theory**: Cross-correlation measures similarity between signal and template:
$$r_{xy}[n] = \sum_{m=-\infty}^{\infty} x[m] \cdot y[m+n]$$

High correlation indicates QRS complex location.

In [ ]:
# Detect R-peaks
r_peaks = preprocessor.detect_r_peaks_correlation(filtered_ecg)

# Calculate HRV
hrv_metrics = preprocessor.calculate_hrv(r_peaks)

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(12, 4))

ax.plot(t, filtered_ecg, 'g-', linewidth=0.8, label='Filtered ECG')
ax.plot(r_peaks / fs, filtered_ecg[r_peaks], 'ro', markersize=10, label='R-peaks')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title(f'R-Peak Detection (Heart Rate: {hrv_metrics["mean_hr_bpm"]:.1f} BPM)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Lab Exp 2: Cross-correlation R-peak detection")
print(f"  Detected {len(r_peaks)} R-peaks")
print(f"  Mean HR: {hrv_metrics['mean_hr_bpm']:.1f} BPM")
print(f"  SDNN: {hrv_metrics['sdnn_ms']:.1f} ms")

## 4. Unit 2: Frequency Domain Analysis <a name="unit2"></a>

### 4.1 DFT vs FFT Comparison (Lab Exp 3-4, Unit 2.3-2.4)

**DFT Formula**:
$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j2\pi kn/N}$$

**Complexity**: DFT is $O(N^2)$, FFT is $O(N \log N)$

In [ ]:
# Initialize frequency analyzer
analyzer = FrequencyAnalyzer(sampling_rate=fs)

# Compare DFT vs FFT performance
perf_results = analyzer.compare_dft_fft(filtered_ecg[:512])

# Visualize performance
methods = ['DFT', 'Custom FFT', 'NumPy FFT']
times = [
    perf_results['dft_time'],
    perf_results['fft_custom_time'],
    perf_results['fft_numpy_time']
]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
bars = ax.bar(methods, times, color=['#ff6b6b', '#4ecdc4', '#45b7d1'])
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title('DFT vs FFT Performance Comparison (N=512)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, time in zip(bars, times):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{time:.4f}s', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("✓ Lab Exp 3-4: DFT and FFT implemented and compared")
print(f"  Speedup (Custom FFT): {perf_results['speedup_custom']:.1f}x")
print(f"  Speedup (NumPy FFT): {perf_results['speedup_numpy']:.1f}x")

### 4.2 Spectral Analysis (Unit 2.2)

**Parseval's Theorem**: Total power in time domain equals total power in frequency domain:
$$\sum_{n} |x[n]|^2 = \frac{1}{N} \sum_{k} |X[k]|^2$$

In [ ]:
# Compute magnitude spectrum
frequencies, magnitude = analyzer.compute_magnitude_spectrum(filtered_ecg)
frequencies_power, power = analyzer.compute_power_spectrum(filtered_ecg)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(frequencies, magnitude, 'r-', linewidth=0.8)
axes[0].set_xlim([0, 50])
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Magnitude')
axes[0].set_title('Magnitude Spectrum')
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(frequencies_power, power, 'g-', linewidth=0.8)
axes[1].set_xlim([0, 50])
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Power (log scale)')
axes[1].set_title('Power Spectral Density')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Unit 2.2: Spectral analysis with Parseval's theorem")

### 4.3 Wavelet Transform (Unit 2.5, Lab Exp 8)

**Theory**: Wavelet transform provides time-frequency localization, useful for analyzing non-stationary signals like ECG.

In [ ]:
# Apply DWT
approximation, details = analyzer.apply_dwt(filtered_ecg, wavelet='db4', level=5)

# Visualize decomposition
fig = analyzer.plot_wavelet_decomposition(filtered_ecg, wavelet='db4', level=5, duration=5.0)
plt.show()

# Denoise using wavelets
denoised = analyzer.denoise_with_dwt(filtered_ecg)

print("✓ Unit 2.5 & Lab Exp 8: Wavelet transform applied")

## 5. Units 5-6: Image Processing on Scalograms <a name="units5-6"></a>

### 5.1 Scalogram Generation

We convert the 1D ECG signal to a 2D time-frequency representation (scalogram) and treat it as a grayscale image.

In [ ]:
# Initialize image processor
image_processor = ScalogramImageProcessor()

# Create scalogram
scalogram, scalogram_freqs = image_processor.create_scalogram(filtered_ecg, fs=fs)
scalogram_image = image_processor.scalogram_to_uint8(scalogram)

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
im = ax.imshow(scalogram, aspect='auto', cmap='viridis', extent=[0, duration, 0, 50])
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('ECG Scalogram (CWT)')
plt.colorbar(im, ax=ax, label='Magnitude')
plt.tight_layout()
plt.show()

print(f"✓ Scalogram generated: {scalogram.shape}")

### 5.2 Histogram Equalization (Unit 5.1, Lab Exp 7)

**Theory**: Histogram equalization redistributes pixel intensities to enhance contrast.

In [ ]:
# Apply histogram equalization
equalized = image_processor.histogram_equalization(scalogram_image)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(scalogram_image, cmap='gray', aspect='auto')
axes[0, 0].set_title('Original Scalogram')
axes[0, 0].axis('off')

axes[0, 1].imshow(equalized, cmap='gray', aspect='auto')
axes[0, 1].set_title('Histogram Equalized')
axes[0, 1].axis('off')

hist_orig, bins_orig = image_processor.compute_histogram(scalogram_image)
axes[1, 0].plot(bins_orig, hist_orig, 'b-')
axes[1, 0].set_title('Original Histogram')
axes[1, 0].set_xlabel('Intensity')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(True, alpha=0.3)

hist_eq, bins_eq = image_processor.compute_histogram(equalized)
axes[1, 1].plot(bins_eq, hist_eq, 'r-')
axes[1, 1].set_title('Equalized Histogram')
axes[1, 1].set_xlabel('Intensity')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Unit 5.1 & Lab Exp 7: Histogram equalization")

### 5.3 Edge Detection (Unit 6, Lab Exp 10)

**Sobel Operator**:
$$G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}, \quad
G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}$$

Edge magnitude: $|G| = \sqrt{G_x^2 + G_y^2}$

In [ ]:
# Apply edge detection
edges_sobel, gx, gy = image_processor.sobel_edge_detection(scalogram_image)
edges_prewitt = image_processor.prewitt_edge_detection(scalogram_image)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(scalogram_image, cmap='gray', aspect='auto')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(edges_sobel, cmap='gray', aspect='auto')
axes[1].set_title('Sobel Edge Detection')
axes[1].axis('off')

axes[2].imshow(edges_prewitt, cmap='gray', aspect='auto')
axes[2].set_title('Prewitt Edge Detection')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("✓ Unit 6 & Lab Exp 10: Edge detection (Sobel, Prewitt)")

### 5.4 Otsu's Thresholding (Unit 6)

**Theory**: Otsu's method finds the optimal threshold that maximizes between-class variance.

In [ ]:
# Apply Otsu's thresholding
binary, threshold = image_processor.otsu_thresholding(scalogram_image)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(scalogram_image, cmap='gray', aspect='auto')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(binary, cmap='gray', aspect='auto')
axes[1].set_title(f"Otsu's Thresholding (T={threshold:.1f})")
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"✓ Unit 6: Otsu's thresholding (threshold={threshold:.1f})")

## 6. Feature Extraction & Classification <a name="features"></a>

### 6.1 Extract Features

In [ ]:
# Initialize feature extractor
feature_extractor = ECGFeatureExtractor(sampling_rate=fs)

# Extract all features
features = feature_extractor.extract_all_features(filtered_ecg, r_peaks, scalogram)

# Display some features
print("\nExtracted Features (sample):")
for i, (key, value) in enumerate(list(features.items())[:10]):
    print(f"  {key}: {value:.4f}")

print(f"\nTotal features: {len(features)}")

### 6.2 Classification

In [ ]:
# Initialize classifier
classifier = ECGClassifier()

# Classify using threshold-based rules
classification, rule_triggers = classifier.classify_threshold_based(features)

print(f"\nClassification Result: {classification}")
print("\nRule Triggers:")
for rule, triggered in rule_triggers.items():
    status = "✓" if triggered else " "
    print(f"  [{status}] {rule}")

## 7. Complete Pipeline <a name="pipeline"></a>

Now let's run the complete end-to-end pipeline:

In [ ]:
# Import main pipeline
from main import ECGAnalysisPipeline

# Initialize pipeline
pipeline = ECGAnalysisPipeline(sampling_rate=360.0)

# Load data (will use synthetic if MIT-BIH unavailable)
ecg_signal, metadata = pipeline.load_mit_bih_record(record_id='100', duration=10.0)

# Run complete analysis
results = pipeline.run_complete_analysis(ecg_signal, visualize=False)

print("\n" + "="*70)
print("ANALYSIS SUMMARY")
print("="*70)
print(f"Classification: {results['classification']}")
print(f"Heart Rate: {results.get('hrv_metrics', {}).get('mean_hr_bpm', 'N/A'):.1f} BPM")
print(f"R-peaks detected: {len(results['r_peaks'])}")
print(f"Total features: {len(results['features'])}")
print("="*70)

## 8. Conclusion <a name="conclusion"></a>

### What We Learned

1. **Unit 1**: Signal sampling, reconstruction, and filtering using convolution
2. **Unit 2**: DFT/FFT algorithms and their performance comparison
3. **Unit 3**: Efficient filtering using Overlap-Add method
4. **Units 5-6**: Image processing techniques on scalograms
5. **Integration**: Complete ECG analysis pipeline

### Key Takeaways

- **FFT is much faster** than DFT (50-2500x speedup)
- **Wavelet transforms** provide excellent time-frequency localization
- **Image processing** techniques can be applied to signal scalograms
- **Feature extraction** combines time, frequency, and texture information

### Syllabus Coverage

✅ All required theory units (1, 2, 3, 5, 6)  
✅ All required lab experiments (1, 2, 3, 4, 7, 8, 9, 10)  
✅ Practical application to real biomedical signals

---

**Thank you for following this tutorial!**

For more information, see:
- `README.md` - Setup and usage guide
- `REPORT.md` - Detailed technical report
- Individual module files for implementation details